[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevisback/bda-course/blob/main/wise2627/notebooks/08_dashboard.ipynb)

# Sitzung 8 — Alles zusammengesetzt: das Insight-Dashboard 🎉

**Big Data Analytics (W3-BDA) · HTW Berlin · Master WI — MEILENSTEIN**

Heute setzen wir alles zusammen, was ihr gebaut habt: Klassifikation, Themen, Aggregation, Zeitverlauf — zu **einem fertigen Dashboard**. Genau das Werkzeug, das ihr in Sitzung 1 gesehen habt. Diesmal gebaut von euch.

> 💡 Kein neuer Stoff heute — nur zusammenfügen und genießen. Dieses Notebook ist eure **Vorlage für die Prüfung**.

## 0. Setup & Daten

In [ ]:
import random, re
from collections import Counter, defaultdict
from datetime import date, timedelta
import matplotlib.pyplot as plt
print('Fertig.')

In [ ]:
import random
from datetime import date, timedelta

SEED = 42
PRODUCT = "Nimbus Q2"

POS = ["der Klang ist hervorragend", "satte Bässe", "die Geräuschunterdrückung ist top",
       "der Akku hält den ganzen Tag", "sitzt super bequem", "Bluetooth verbindet sofort",
       "top verarbeitet", "klasse für den Preis", "die App ist übersichtlich",
       "die Passform ist perfekt", "der Sound ist klar und ausgewogen"]
NEG = ["die App stürzt ständig ab", "der rechte Ohrhörer lädt nicht mehr",
       "die Geräuschunterdrückung rauscht", "viel zu teuer", "die Touch-Steuerung reagiert kaum",
       "das Case wirkt billig", "der Akku ist nach einer Stunde leer",
       "die Verbindung bricht ab", "sie fallen leicht aus dem Ohr", "der Bass ist matschig"]

POS_OPENERS = ["Bin begeistert:", "Wirklich gut:", "Kann ich empfehlen –", "Top Kauf.",
               "Sehr zufrieden:", "Absolute Kaufempfehlung.", "Ich liebe sie:",
               "Klare Sache:", "Rundum gelungen:", "Volle Punktzahl:", "Endlich zufrieden:",
               "Was soll ich sagen –", "Genau richtig:", "Bestellung hat sich gelohnt:"]
NEG_OPENERS = ["Enttäuschend:", "Leider schlecht:", "Finger weg –", "Bin frustriert:",
               "Nicht zu empfehlen.", "Schade um das Geld:", "Ärgerlich:",
               "Reklamiert:", "Bin raus:", "Nie wieder:", "Herbe Enttäuschung:",
               "Das war nichts:", "Zurückgeschickt:", "Vorsicht:"]
NEU_TEMPLATES = ["Ganz okay, {a}, aber nichts Besonderes.",
                 "Erfüllt seinen Zweck. {a_cap}.",
                 "Durchschnittlich. {a_cap}, mehr nicht.",
                 "Habe sie seit Kurzem, {a} – kann noch nicht viel sagen."]
SARCASTIC = ["Super, schon nach drei Tagen kaputt. Echt klasse Qualität.",
             "Toll, dass die App jedes Mal abstürzt. Genau das wollte ich.",
             "Wunderbar, 200 Euro für Ohrhörer, die nach links driften. Ein Traum.",
             "Ganz großes Kino, der Akku hält sagenhafte 40 Minuten.",
             "Klasse, nach einer Woche nur noch Rauschen. Wirklich durchdacht.",
             "Perfekt, der linke fällt ständig raus. Genau mein Wunsch.",
             "Herrlich, die Verbindung bricht alle fünf Minuten ab. Danke auch.",
             "Sensationell, das Case bricht beim ersten Öffnen. Qualität eben.",
             "Bravo, nach dem Update ist die Hälfte der Funktionen weg.",
             "Fantastisch leise – weil nach zwei Tagen einfach tot."]
FAKE = ["BESTES PRODUKT EVER!!! Kauft bei www.super-deals-guenstig.example!!!",
        "5 Sterne 5 Sterne bester shop schnelle lieferung AAA+++",
        "Gutschein Code NIMBUS100 auf meiner Seite jetzt klicken!!!",
        "amazing product best quality buy now discount link in profile",
        "TOP TOP TOP unbedingt kaufen billigster preis hier klicken",
        "gratis versand nur heute!!! rabattcode DEAL22 einlösen!!!",
        "beste kopfhoerer der welt jetzt zuschlagen link im profil",
        "WOW einfach WOW kaufen kaufen kaufen bester preis garantiert",
        "unglaublich guenstig hier klicken und sparen sparen sparen",
        "mega angebot heute -70% nur ueber meinen link!!!"]
ENGLISH = [("Sound quality is great but the app is a disaster.", "mixed"),
           ("Battery life is amazing, best earbuds I have owned.", "positive"),
           ("Stopped working after a week, very disappointed.", "negative"),
           ("Comfortable fit and clear sound, happy with the purchase.", "positive"),
           ("The noise cancelling is weak and the case feels cheap.", "negative")]
JUNK = ["", "   ", ".", "???", "kein kommentar", "-", "n/a", "...", "!!", "??", "keine angabe", "test"]

def _pos(rng):
    o = rng.choice(POS_OPENERS); a = rng.sample(POS, rng.choice([1, 2]))
    return f"{o} {' und '.join(a)}.", "positive"
def _neg(rng):
    o = rng.choice(NEG_OPENERS); a = rng.sample(NEG, rng.choice([1, 2]))
    return f"{o} {' und '.join(a)}.", "negative"
def _mixed(rng):
    p = rng.choice(POS); n = rng.choice(NEG)
    conn = rng.choice([" - aber ", ", allerdings ", ". Leider ", ", jedoch "])
    return f"{p[0].upper()+p[1:]}{conn}{n}.", "mixed"
def _neutral(rng):
    a = rng.choice(POS + NEG)
    return rng.choice(NEU_TEMPLATES).format(a=a, a_cap=a[0].upper()+a[1:]), "neutral"
def _rating_for(truth, rng):
    return rng.choice({"positive":[4,5,5],"negative":[1,1,2],"mixed":[2,3,4],
                       "neutral":[3,3,4],"fake":[5,5]}.get(truth,[1,3,5]))

def generate(n=300, seed=SEED):
    rng = random.Random(seed)
    start = date(2026, 1, 1)
    rows = []
    for i in range(n):
        r = rng.random()
        if r < 0.34:   text, truth = _pos(rng)
        elif r < 0.60: text, truth = _neg(rng)
        elif r < 0.72: text, truth = _mixed(rng)
        elif r < 0.80: text, truth = _neutral(rng)
        elif r < 0.88: text, truth = rng.choice(SARCASTIC), "negative"
        elif r < 0.93: text, truth = rng.choice(FAKE), "fake"
        elif r < 0.98: text, truth = rng.choice(ENGLISH)
        else:          text, truth = rng.choice(JUNK), "junk"
        day = int(rng.triangular(0, 270, 200 if truth == "negative" else 90))
        rows.append({
            "review_id": f"R{i:04d}",
            "date": (start + timedelta(days=day)).isoformat(),
            "product": PRODUCT,
            "rating": _rating_for(truth, rng),
            "text": text,
            "true_sentiment": truth,
            "is_sarcastic": text in SARCASTIC,
            "is_fake": text in FAKE,
        })
    if n > 20:
        for j, src in enumerate([5, 12, 30]):
            rows.append(dict(rows[src], review_id=f"R{n+j:04d}"))
    rng.shuffle(rows)
    return rows

## 1. Die Pipeline — alles, was ihr gebaut habt

Eine Zelle, die alle Bausteine der letzten Wochen enthält: klassifizieren, Themen erkennen, aggregieren. (Mock = key-frei; echt per Schalter.)

In [ ]:
# --- Klassifikation (Sitzung 2/3) ---
def classify(text):
    t=(text or '').lower()
    pos=sum(w in t for w in ['gut','top','super','toll','hervorragend','bequem',
            'stabil','klasse','liebe','perfekt','begeistert','gelungen'])
    neg=sum(w in t for w in ['schlecht','kaputt','teuer','stürzt','rauscht','billig',
            'nervt','enttäuscht','leer','bricht','matschig','finger weg'])
    if pos and neg: return 'mixed'
    if pos: return 'positive'
    if neg: return 'negative'
    return 'neutral'

# --- Themen (Sitzung 5) ---
STICHWORT={'Klang':['klang','sound','bass','rauscht'],'Akku':['akku','laden','leer','batterie'],
           'App':['app'],'Preis':['preis','teuer','euro'],'Komfort':['bequem','sitzt','ohr','passform'],
           'Verbindung':['bluetooth','verbindung','bricht']}
def themen(t):
    t=t.lower(); return [k for k,ws in STICHWORT.items() if any(w in t for w in ws)]

# --- Pipeline: rohe Reviews -> angereicherte Records ---
def pipeline(reviews):
    out=[]
    for r in reviews:
        out.append({**r, 'sentiment': classify(r['text']), 'themen': themen(r['text'])})
    return out

reviews = generate(300)
daten = pipeline(reviews)
print(f'{len(daten)} Bewertungen durch die Pipeline.')

## 2. Aggregieren — die Kennzahlen fürs Dashboard

In [ ]:
def aggregiere(daten):
    n=len(daten)
    sentiment=Counter(d['sentiment'] for d in daten)
    lob=Counter(); kritik=Counter()
    for d in daten:
        for th in d['themen']:
            if d['sentiment']=='positive': lob[th]+=1
            if d['sentiment']=='negative': kritik[th]+=1
    monat=defaultdict(lambda: Counter())
    for d in daten:
        monat[d['date'][:7]][d['sentiment']]+=1
    return {'n':n,'sentiment':dict(sentiment),'lob':lob.most_common(5),
            'kritik':kritik.most_common(5),'monat':dict(sorted(monat.items()))}

agg = aggregiere(daten)
print('Sentiment:', agg['sentiment'])
print('Top-Kritik:', agg['kritik'][:3])

## 3. Die Executive Summary

Eine Entscheiderin liest keine Tabellen — sie will *einen Absatz*. Wir bauen die Zusammenfassung **aus den Zahlen** (deterministisch, key-frei):

In [ ]:
def summary(agg, produkt='Nimbus Q2'):
    n=agg['n']; s=agg['sentiment']
    pos=round(100*s.get('positive',0)/n); neg=round(100*s.get('negative',0)/n)
    top_lob = agg['lob'][0][0] if agg['lob'] else '-'
    top_kritik = agg['kritik'][0][0] if agg['kritik'] else '-'
    return (f'Von {n} Bewertungen zum {produkt} sind rund {pos}% positiv und '
            f'{neg}% negativ. Am meisten gelobt wird {top_lob!r}, '
            f'häufigster Kritikpunkt ist {top_kritik!r}.')

print(summary(agg))

## 4. Das Dashboard

Jetzt das visuelle Gesamtbild — sauber, nicht verspielt. Zwei Panels: Sentiment-Verteilung und Zeitverlauf.

In [ ]:
def dashboard(agg, produkt='Nimbus Q2'):
    print('='*56)
    print(f'  REVIEW RADAR  —  {produkt}   ({agg["n"]} Bewertungen)')
    print('='*56)
    print('\n' + summary(agg) + '\n')
    print('  TOP-LOB     :', ', '.join(f'{t} ({c})' for t,c in agg['lob'][:3]))
    print('  TOP-KRITIK  :', ', '.join(f'{t} ({c})' for t,c in agg['kritik'][:3]))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13,4))
    order=['positive','mixed','neutral','negative']
    farben={'positive':'#1f9d55','mixed':'#f6993f','neutral':'#a0aec0','negative':'#e3342f'}
    werte=[agg['sentiment'].get(s,0) for s in order]
    ax1.bar(order, werte, color=[farben[s] for s in order])
    ax1.set_title('Sentiment-Verteilung'); ax1.set_ylabel('Anzahl')
    monate=list(agg['monat'])
    pos=[agg['monat'][m].get('positive',0) for m in monate]
    neg=[agg['monat'][m].get('negative',0) for m in monate]
    ax2.plot(monate,pos,marker='o',color='#1f9d55',label='Positiv')
    ax2.plot(monate,neg,marker='o',color='#e3342f',label='Negativ')
    ax2.set_title('Sentiment im Zeitverlauf'); ax2.legend()
    ax2.tick_params(axis='x', rotation=45)
    plt.tight_layout(); plt.show()

dashboard(agg)

> 🎉 **Das ist es.** Von rohen Bewertungen zu einem klaren Bild — das Werkzeug aus Sitzung 1, gebaut von euch. **Das ist die Vorlage für eure Prüfung.**

## 5. Eure Aufgabe

Lasst die Pipeline auf einer **Teilmenge** laufen (z. B. nur ein Quartal) und baut das Dashboard dafür. Das Werkzeug funktioniert für jeden Ausschnitt.

In [ ]:
# Beispiel: nur die zweite Jahreshälfte
teilmenge = [d for d in daten if d['date'] >= '2026-07']
print(f'{len(teilmenge)} Bewertungen ab Juli')
dashboard(aggregiere(teilmenge))

## 6. Eine unbequeme Frage zum Schluss

Die Executive Summary oben klingt überzeugend. Aber: **stimmt sie überhaupt?** Unsere Version ist aus den Zahlen gebaut — die *kann* nicht lügen. Aber ein **echtes LLM**, das die Zusammenfassung schreibt, könnte Dinge behaupten, die in den Daten gar nicht stehen.

> 🎓 **Live (Vortragende:r):** Die Summary vom echten LLM schreiben lassen und Satz für Satz gegen die Zahlen prüfen. Erfindet es etwas? Übertreibt es?

> ⚠️ **Kernpunkt & Ausblick:** Ein Dashboard *sieht* überzeugend aus — das heißt nicht, dass es stimmt. Genau da geht es nächste Woche weiter: **Wann lügt das LLM?** (Sitzung 9). Die erste Hälfte des Kurses war *bauen*. Ab jetzt lernen wir, dem Gebauten zu **misstrauen**.

## Geschafft — Halbzeit! 🎉

Ihr habt ein vollständiges KI-Insight-Produkt gebaut. Ab Sitzung 9 drehen wir den Spieß um: nicht mehr *bauen*, sondern *prüfen* — wo das Werkzeug lügt, wie man es misst, und wie man ihm am Ende trauen kann.